In [1]:
import pandas as pd

In [37]:
x=pd.read_csv("kdd_train.csv")
print(set(x['labels']))

{'pod', 'normal', 'multihop', 'perl', 'teardrop', 'portsweep', 'guess_passwd', 'ftp_write', 'land', 'nmap', 'loadmodule', 'phf', 'buffer_overflow', 'smurf', 'back', 'rootkit', 'ipsweep', 'spy', 'imap', 'warezclient', 'satan', 'neptune', 'warezmaster'}


In [3]:
x["labels"] = x["labels"].replace(['neptune', 'warezclient', 'ipsweep', 'portsweep',
       'teardrop', 'nmap', 'satan', 'smurf', 'pod', 'back',
       'guess_passwd', 'ftp_write', 'multihop', 'rootkit',
       'buffer_overflow', 'imap', 'warezmaster', 'phf', 'land',
       'loadmodule', 'spy', 'perl'], 'attack')

In [4]:
x["labels"].unique()

array(['normal', 'attack'], dtype=object)

In [5]:
x['labels'] = x['labels'].map({'normal': 0, 'attack': 1})

In [6]:
x["labels"].unique()

array([0, 1])

In [7]:
x.columns

Index(['duration', 'protocol_type', 'service', 'flag', 'src_bytes',
       'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
       'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
       'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
       'num_access_files', 'num_outbound_cmds', 'is_host_login',
       'is_guest_login', 'count', 'srv_count', 'serror_rate',
       'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
       'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
       'dst_host_srv_count', 'dst_host_same_srv_rate',
       'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
       'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
       'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
       'dst_host_srv_rerror_rate', 'labels'],
      dtype='object')

In [40]:
X = x.iloc[:, :-1].values

In [44]:
y = x.iloc[:, 41].values

In [45]:
#encoding categorical data
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

#first applying label encoding to convert strings to number.
#display(x[:,2])
labelencoder_x_1 = LabelEncoder()
labelencoder_x_2 = LabelEncoder()
labelencoder_x_3 = LabelEncoder()
X[:, 1] = labelencoder_x_1.fit_transform(X[:, 1])
X[:, 2] = labelencoder_x_2.fit_transform(X[:, 2])
X[:, 3] = labelencoder_x_3.fit_transform(X[:, 3])

['tcp' 'udp' 'tcp' ... 'tcp' 'tcp' 'tcp']
[1 2 1 ... 1 1 1]


In [11]:
from sklearn.model_selection import train_test_split
import numpy as np
from collections import Counter

In [66]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
print('Original dataset shape {}'.format(Counter(y)))
print('Training dataset shape {}'.format(Counter(y_train)))

Original dataset shape Counter({'normal': 67343, 'neptune': 41214, 'satan': 3633, 'ipsweep': 3599, 'portsweep': 2931, 'smurf': 2646, 'nmap': 1493, 'back': 956, 'teardrop': 892, 'warezclient': 890, 'pod': 201, 'guess_passwd': 53, 'buffer_overflow': 30, 'warezmaster': 20, 'land': 18, 'imap': 11, 'rootkit': 10, 'loadmodule': 9, 'ftp_write': 8, 'multihop': 7, 'phf': 4, 'perl': 3, 'spy': 2})
Training dataset shape Counter({'normal': 45150, 'neptune': 27637, 'satan': 2427, 'ipsweep': 2375, 'portsweep': 1977, 'smurf': 1785, 'nmap': 981, 'back': 646, 'teardrop': 608, 'warezclient': 573, 'pod': 131, 'guess_passwd': 32, 'buffer_overflow': 19, 'warezmaster': 12, 'land': 11, 'loadmodule': 8, 'multihop': 6, 'rootkit': 6, 'imap': 6, 'ftp_write': 5, 'phf': 3, 'perl': 2, 'spy': 1})


In [13]:
from imblearn.datasets import make_imbalance
X_rs, y_rs = make_imbalance(X_train, y_train, sampling_strategy={1: 1000, 0: 65},
                      random_state=0)
print('Random undersampling {}'.format(Counter(y_rs)))

Random undersampling Counter({np.int64(1): 1000, np.int64(0): 65})


In [14]:
X_rs.shape

(1065, 41)

In [15]:
from imblearn.under_sampling import (RandomUnderSampler,
                                     ClusterCentroids,
                                     TomekLinks,
                                     NeighbourhoodCleaningRule,
                                     NearMiss)

In [16]:
#sampler = RandomUnderSampler(sampling_strategy={1: 1000, 0: 1000})
sampler = RandomUnderSampler()
X_rs1, y_rs1 = sampler.fit_resample(X_train, y_train)
print('Random undersampling {}'.format(Counter(y_rs1)))



Random undersampling Counter({np.int64(0): 39251, np.int64(1): 39251})


In [17]:
X_rs1.shape

(78502, 41)

In [18]:
sampler = ClusterCentroids(sampling_strategy={1: 1000, 0: 1000})
X_rs2, y_rs2 = sampler.fit_resample(X_train, y_train)
print('Cluster centriods undersampling {}'.format(Counter(y_rs2)))


Cluster centriods undersampling Counter({np.int64(0): 1000, np.int64(1): 1000})


In [19]:
X_rs2.shape

(2000, 41)

In [20]:
sampler = TomekLinks()
X_rs3, y_rs3 = sampler.fit_resample(X_train, y_train)
print('TomekLinks undersampling {}'.format(Counter(y_rs3)))

TomekLinks undersampling Counter({np.int64(0): 45118, np.int64(1): 39251})


In [21]:
X_rs3.shape

(84369, 41)

In [22]:
sampler = NeighbourhoodCleaningRule()
X_rs4, y_rs4 = sampler.fit_resample(X_train, y_train)
print('NearestNeighbours Clearning Rule undersampling {}'.format(Counter(y_rs4)))


NearestNeighbours Clearning Rule undersampling Counter({np.int64(0): 44724, np.int64(1): 39251})


In [23]:
X_rs4.shape

(83975, 41)

In [24]:
sampler = NearMiss()
X_rs5, y_rs5 = sampler.fit_resample(X_train, y_train)
print('NearMiss{}'.format(Counter(y_rs5)))


NearMissCounter({np.int64(0): 39251, np.int64(1): 39251})


In [25]:
X_rs5.shape

(78502, 41)

In [26]:
from imblearn.over_sampling import (RandomOverSampler,
                                    SMOTE,
                                    ADASYN)

In [27]:
# RandomOverSampler
  # With over-sampling methods, the number of samples in a class
  # should be greater or equal to the original number of samples.
#sampler = RandomOverSampler(sampling_strategy={1: 46000, 0:46000})
sampler = RandomOverSampler()
X_rs6, y_rs6 = sampler.fit_resample(X_train, y_train)
print('RandomOverSampler {}'.format(Counter(y_rs6)))


RandomOverSampler Counter({np.int64(1): 45150, np.int64(0): 45150})


In [28]:
X_rs6.shape

(90300, 41)

In [29]:
sampler = SMOTE()
X_rs7, y_rs7 = sampler.fit_resample(X_train, y_train)
print('SMOTE {}'.format(Counter(y_rs7)))


SMOTE Counter({np.int64(1): 45150, np.int64(0): 45150})


In [30]:
X_rs7.shape

(90300, 41)

In [31]:
sampler = ADASYN()
X_rs8, y_rs8 = sampler.fit_resample(X_train, y_train)
print('ADASYN {}'.format(Counter(y_rs8)))


ADASYN Counter({np.int64(1): 45227, np.int64(0): 45150})


In [32]:
X_rs8.shape

(90377, 41)